# `test_name_inference.py` 및 미매칭 상품군 분석

이 노트북은 키워드 추론 모델의 성능 평가를 위한 GT 풀 분석과, `batch_attr_infer_unmatched.py`에서 다루는 **블로그 미매칭 신제품(추론 대상)** 리스트를 분석하기 위해 작성되었습니다.

또한, 사용자가 별도로 정리한 `unmatched_products.xlsx`의 '살려' 항목을 기반으로 **네이버 통합 검색 API**를 활용한 최신 데이터 수집 로직을 포함합니다.

In [ ]:
import os
import sys
import pandas as pd
import random
import re
import requests
import time
from datetime import datetime
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import display

# .env 파일 로드 (네이버 API 키 등)
load_dotenv()

# 프로젝트 루트 경로 설정
notebook_dir = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(notebook_dir, "..", ".."))

## 1. 블로그 매칭 데이터 (Ground Truth) 분석

In [ ]:
BLOG_KEYWORDS_FILE = os.path.join(PROJECT_ROOT, "data", "processed", "blog_with_keywords_filtered.csv")
df_blog = pd.read_csv(BLOG_KEYWORDS_FILE, encoding="utf-8-sig")
gt_clean = df_blog[df_blog["review_keywords"].notna()].drop_duplicates("product_name").copy()
print(f"검증된 블로그 상품군: {len(gt_clean)}종")

## 2. 미매칭 신제품 (Inference 대상) 분석

In [ ]:
FILTER_LIST_FILE = os.path.join(PROJECT_ROOT, "data", "processed", "블로그크롤링", "00_TRUE_NPD_LIST_FILTERED.xlsx")
if os.path.exists(FILTER_LIST_FILE):
    filter_df = pd.read_excel(FILTER_LIST_FILE)
    name_col = [c for c in filter_df.columns if "ITEM_NM" in c][0]
    target_products = set(filter_df[name_col].dropna().unique())
    matched_products = set(df_blog["product_name"].dropna().unique())
    unmatched_names = sorted(target_products - matched_products)
    print(f"미매칭 (추론 대상): {len(unmatched_names)}종")
else:
    print(f"파일을 찾을 수 없습니다: {FILTER_LIST_FILE}")

## 3. 재수집 대상 분석 ('살려' 항목)

In [ ]:
def clean_product_name(name):
    if not isinstance(name, str): return ""
    prefixes = [r"^APP예약\)", r"^APP\)", r"^PB\)", r"^濡\)", r"^곗\)", r"^\)", r"^臾댁\)", r"^援щⅤ硫\)", r"^25異\)", r"^GS\)", r"^CU\)", r"^7-11\)"]
    cleaned = name
    for p in prefixes: cleaned = re.sub(p, "", cleaned)
    cleaned = re.sub(r"\d+(g|ml|kg|l|L|P|EA|媛|)($|\s|\))", "", cleaned)
    cleaned = re.sub(r"\(\d+.*\)", "", cleaned)
    cleaned = re.sub(r"[\(\)\[\]]", " ", cleaned)
    return re.sub(r"\s+", " ", cleaned).strip()

UNMATCHED_XLSX = os.path.join(PROJECT_ROOT, "unmatched_products.xlsx")
if os.path.exists(UNMATCHED_XLSX):
    unmatched_raw = pd.read_excel(UNMATCHED_XLSX)
    revive_df = unmatched_raw[unmatched_raw['살려'].astype(str).str.contains('O', na=False, case=False)].copy()
    revive_df['cleaned_name'] = revive_df['product_name'].apply(clean_product_name)
    unique_revive = revive_df.drop_duplicates('cleaned_name').copy()
    print(f"재수집 대상 유니크 제품 수: {len(unique_revive)}개")
    display(unique_revive[['product_name', 'cleaned_name']].head(10))
else:
    print(f"파일을 찾을 수 없습니다: {UNMATCHED_XLSX}")

## 4. 네이버 통합 검색 및 전수 수집 실행

확장된 채널 리스트를 기반으로 '살려' 항목 전체에 대한 본문 수집을 진행합니다.

In [ ]:
# 확장된 수집 허용 채널 리스트
ALLOWED_CHANNELS = [
    "blog.naver.com",   # 네이버 블로그
    "cafe.naver.com",   # 네이버 카페
    "kin.naver.com",    # 지식iN
    "post.naver.com",   # 네이버 포스트
    "tistory.com",      # 티스토리
    "news.naver.com",   # 네이버 뉴스
    "brunch.co.kr"      # 브런치
]

def fetch_full_content(url):
    try:
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
        if "blog.naver.com" in url and "PostView.naver" not in url:
            parts = url.split("blog.naver.com/")[1].split("/")
            if len(parts) >= 2: url = f"https://blog.naver.com/PostView.naver?blogId={parts[0]}&logNo={parts[1].split('?')[0]}"
        
        res = requests.get(url, headers=headers, timeout=7)
        soup = BeautifulSoup(res.text, 'html.parser')
        for s in soup(['script', 'style', 'header', 'footer', 'nav', 'aside']): s.decompose()
        content = soup.select_one('.se-main-container') or soup.select_one('#postViewArea') or soup.select_one('article')
        if not content: content = soup.body
        return content.get_text(separator='\n', strip=True) if content else ""
    except: return ""

def search_naver_integrated(query, display_count=10):
    client_id = os.getenv("NAVER_CLIENT_ID")
    client_secret = os.getenv("NAVER_CLIENT_SECRET")
    if not client_id: return []
    url = "https://openapi.naver.com/v1/search/webkr.json"
    headers = {"X-Naver-Client-Id": client_id, "X-Naver-Client-Secret": client_secret}
    params = {"query": query, "display": display_count}
    try:
        res = requests.get(url, headers=headers, params=params)
        return res.json().get('items', [])
    except: return []

def run_revive_full_crawler(target_df, allowed_channels=None, limit_per_product=3):
    results = []
    total = len(target_df)
    
    print(f"🚀 전수 수집 시작 (총 {total}종)... 채널 필터: {len(allowed_channels) if allowed_channels else '전체'}")
    
    for i, row in enumerate(target_df.itertuples(), start=1):
        name = row.cleaned_name
        orig_name = row.product_name
        
        print(f"[{i}/{total}] 🔍 '{name}' 수집 중... ", end="")
        
        # 2단계 검색 전략
        items = search_naver_integrated(f"세븐일레븐 {name}", display_count=10)
        if not items: 
            print("(2단계 시도) ", end="")
            items = search_naver_integrated(name, display_count=10)
            
        count = 0
        for item in items:
            if allowed_channels and not any(domain in item['link'] for domain in allowed_channels): continue
            
            full_text = fetch_full_content(item['link'])
            content = full_text if len(full_text) > 50 else item['description'].replace('<b>','').replace('</b>','')
            
            results.append({
                "product_name": orig_name,
                "search_keyword": name,
                "title": item['title'].replace('<b>','').replace('</b>',''),
                "content": content,
                "link": item['link']
            })
            count += 1
            if count >= limit_per_product: break
            
        print(f"-> {count}건 확보")
        time.sleep(0.1)
        
        # 중간 저장 (50건마다)
        if i % 50 == 0:
            pd.DataFrame(results).to_csv("revive_crawled_tmp.csv", index=False, encoding='utf-8-sig')
            print(f"   >>> {i}건 완료 중간 저장 완료")

    return pd.DataFrame(results)

# 실제 전수 실행
if 'unique_revive' in locals():
    final_df = run_revive_full_crawler(unique_revive, allowed_channels=ALLOWED_CHANNELS)
    
    # 결과 요약 및 저장
    if not final_df.empty:
        print(f"\n✅ 최종 수집 완료! 총 {len(final_df)}개 게시글 확보")
        output_path = "revive_crawled_results.csv"
        final_df.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"💾 결과 저장됨: {output_path}")
else:
    print("⚠️ 'unique_revive' 리스트를 먼저 생성해 주세요.")

## 5. 수집 결과 본문 글자 수 분석

수집된 본문 데이터의 길이를 확인하고, 분석에 적합한 적정 길이(4,000자 이내)의 게시물을 확인합니다.

In [ ]:
CRAWLED_RESULTS_FILE = "revive_crawled_results.csv"

if os.path.exists(CRAWLED_RESULTS_FILE):
    df_res = pd.read_csv(CRAWLED_RESULTS_FILE, encoding="utf-8-sig")
    
    # 글자 수 계산
    df_res['char_count'] = df_res['content'].fillna('').apply(len)
    
    print(f"전체 수집 게시물 수: {len(df_res)}개")
    
    # 4,000자 이내 필터링
    df_valid = df_res[df_res['char_count'] <= 4000].copy()
    print(f"4,000자 이내 유효 게시물: {len(df_valid)}개")
else:
    print(f"파일을 찾을 수 없습니다: {CRAWLED_RESULTS_FILE}")

## 6. 분석용 최종 데이터셋 생성 (형식 맞추기)

4,000자 이내 유효 게시물을 기존 `blog_with_keywords_filtered.csv` 형식에 맞춰 엑셀로 저장합니다.
이 데이터는 이후 키워드 추출 파이프라인으로 투입됩니다.

In [ ]:
if 'df_valid' in locals() and not df_valid.empty:
    # 기존 데이터셋 컬럼 구성: 
    # ['중분류', '검색어', 'product_name', '블로그제목', '본문내용', 'review_keywords', 'hin_keywords']
    
    # 1. 컬럼 매핑 및 빈 값 생성
    final_export = pd.DataFrame({
        '중분류': '', # 추후 매핑 필요 시 로직 추가
        '검색어': df_valid['search_keyword'],
        'product_name': df_valid['product_name'],
        '블로그제목': df_valid['title'],
        '본문내용': df_valid['content'],
        'review_keywords': '', # LLM 추출 대상
        'hin_keywords': ''     # LLM 추출 대상
    })
    
    # 2. 엑셀 저장
    output_xlsx = "revive_final_for_extraction.xlsx"
    final_export.to_excel(output_xlsx, index=False, encoding='utf-8-sig')
    
    print(f"✅ 추출 대기용 데이터셋 생성 완료: {output_xlsx}")
    print(f"   (총 {len(final_export)}행)")
    
    # 3. 기존 블로그 키워드 추출 로직 연동 가이드
    print("\n--- 다음 작업 가이드 ---")
    print("1. 위 엑셀 파일을 'data/processed/blog_with_keywords_revive.csv'로 저장합니다.")
    print("2. keyword_extractor.py를 사용하여 빈 키워드 컬럼을 채웁니다.")
    print("3. 기존 blog_with_keywords_filtered.csv와 합칩니다.")
else:
    print("⚠️ 유효한 데이터가 없습니다. 먼저 수집을 진행해 주세요.")